In [ ]:
# Install all required libraries
!pip install -q langchain_groq pypdf langchain_community langchain chromadb groq wikipedia
!pip install -q git+https://github.com/huggingface/parler-tts.git
!pip install -q deep-translator
!pip install -q soundfile==0.12.1 # Fix dependency conflict often seen with coqui-tts
!pip install -q pydantic==2.8.2 # Fix dependency conflict for newer LangChain

In [ ]:
# Import all required libraries
import os
import torch
import time
import json
import wikipedia
import soundfile as sf
import numpy as np
from groq import Groq
from pypdf import PdfReader
from operator import itemgetter
from langchain_community.retrievers import WikipediaRetriever
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from google.colab import drive
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer
from deep_translator import (
    GoogleTranslator,
    MicrosoftTranslator,
    MyMemoryTranslator,
    PonsTranslator,
    LingueeTranslator,
    LibreTranslator,
    DeeplTranslator,
    single_detection,
    batch_detection
)

In [ ]:
# Set device
device = "cuda:0" if torch.cuda.is_available() else "cpu"

In [ ]:
# 1. Mount Google Drive to access your PDF
drive.mount('/content/drive')

In [ ]:
# 2. Set your API Key
# IMPORTANT: Replace the placeholder with os.getenv('GROQ_API_KEY') for security in a real environment
os.environ["GROQ_API_KEY"] = "groq_api_key"

In [ ]:
# 3. Define File Path (Update if your path changed)
PDF_FILE_PATH = "/content/drive/MyDrive/Dataset/Educational_AI_Explainer/Class_10_English_English_Medium-2024_Edition-www.tntextbooks.in.pdf"

In [ ]:
# 4. Create output directory
OUTPUT_DIR = "output"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

print(f"Setup complete. Output files will be saved in: {OUTPUT_DIR}")

In [ ]:
def extract_text(file_path):
    """Extracts text from PDF or TXT files."""
    if file_path.endswith(".pdf"):
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            # Simple check to skip pages with little text (like blank pages)
            page_text = page.extract_text()
            if page_text and len(page_text.strip()) > 50:
                text += page_text
        return text
    elif file_path.endswith(".txt"):
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()
    else:
        raise ValueError("Unsupported file type. Please use PDF or TXT.")

RAG INITIALIZATION

In [ ]:
# 1. Extract and Chunk Text from PDF
pdf_text_data = extract_text(PDF_FILE_PATH)
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=200)
pdf_text_chunks = splitter.split_text(pdf_text_data)
print(f"Split PDF text into {len(pdf_text_chunks)} chunks.")

In [ ]:
# 2. Initialize Embedding Model (to convert text to vectors)
# Using a fast, high-quality multilingual model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
print("Embedding model loaded.")

In [ ]:
# 3. Create/Load ChromaDB Vector Store
# This process takes time, but only needs to be run once.
print("Creating ChromaDB from PDF chunks...")
chroma_db = Chroma.from_texts(
    texts=pdf_text_chunks,
    embedding=embeddings,
    persist_directory="./vectorstore" # Store the vector index locally
)
pdf_retriever = chroma_db.as_retriever(search_kwargs={"k": 2}) # Retrieve top 2 chunks
print("ChromaDB vector store created and ready.")

In [ ]:
# 4. Initialize General Knowledge Retriever (Wikipedia)
wiki_retriever = WikipediaRetriever(top_k_results=1, doc_content_chars_max=1000)
print("Wikipedia retriever initialized.")

In [ ]:
# 5. Initialize LLM
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile", # Updated to a different model
    temperature=0.7
)
print(f"LLM initialized with model: {llm.model_name}.")

TTS (indic-parler)

In [ ]:
# Load ParlerTTS model and tokenizers
# Using 'parler-tts/parler_tts_mini_v0.1' as it's a known working model from the library.
model = ParlerTTSForConditionalGeneration.from_pretrained("parler-tts/parler_tts_mini_v0.1").to(device)
tokenizer = AutoTokenizer.from_pretrained("parler-tts/parler_tts_mini_v0.1")
description_tokenizer = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)

In [ ]:
def chunk_text(text, max_words=50):
    """Split text into chunks of maximum words"""
    words = text.split()
    chunks = []
    current_chunk = []

    for word in words:
        current_chunk.append(word)
        if len(current_chunk) >= max_words:
            chunks.append(" ".join(current_chunk))
            current_chunk = []

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

In [ ]:
def generate_long_audio(model, tokenizer, description_tokenizer, long_text, description, max_words=50):
    """Generate audio for long text by chunking"""
    device = next(model.parameters()).device

    # Prepare description once
    description_input_ids = description_tokenizer(
        description, return_tensors="pt"
    ).to(device)

    # Split long text into chunks
    text_chunks = chunk_text(long_text, max_words=max_words)

    all_audio = []

    for i, chunk in enumerate(text_chunks):
        print(f"Processing chunk {i+1}/{len(text_chunks)}: {chunk[:50]}...")

        # Prepare prompt for current chunk
        prompt_input_ids = tokenizer(chunk, return_tensors="pt").to(device)

        # Generate audio for chunk
        with torch.no_grad():
            generation = model.generate(
                input_ids=description_input_ids.input_ids,
                attention_mask=description_input_ids.attention_mask,
                prompt_input_ids=prompt_input_ids.input_ids,
                prompt_attention_mask=prompt_input_ids.attention_mask
            )

        audio_chunk = generation.cpu().numpy().squeeze()
        all_audio.append(audio_chunk)

        # Add small silence between chunks for natural flow (optional)
        silence_duration = 0.1  # 100ms silence between chunks
        silence_samples = int(silence_duration * model.config.sampling_rate)
        silence = np.zeros(silence_samples)
        all_audio.append(silence)

    # Concatenate all audio chunks
    final_audio = np.concatenate(all_audio)

    return final_audio

In [ ]:
DESCRIPTION = {
    "English" : "The Female speaker Mary with a low-to-moderate pitch and a slightly expressive, clear tone, speaking at a moderate pace. The recording is high quality, close-sounding and a very clear audio. Use a English accent.",
    "Tamil" : "An Indian Female speaker jaya with a low-to-moderate pitch and a slightly expressive, clear tone, speaking at a moderate pace. The recording is high quality, close-sounding and a very clear audio. Use a Tamil accent.",
    "Hindi" : "An Indian Female speaker Divya with a low-to-moderate pitch and a slightly expressive, clear tone, speaking at a moderate pace. The recording is high quality, close-sounding and a very clear audio. Use a Hindi accent."
}

Core Logic: Function and Configuration

In [ ]:
# Maps user input to LLM prompt language codes
LANGUAGE_MAP = {
    "English": "English",
    "Tamil": "Tamil",
    "Hindi": "Hindi"
}

In [ ]:
# Define the expected JSON output format
RESPONSE_JSON = {
    "text_explanation": "The generated educational explanation.",
    "audio_filename": "The relative path to the generated audio file (e.g., output/topic.mp3)."
}

In [ ]:
# Create the LLM Prompt Template
prompt = ChatPromptTemplate.from_messages(
      [
          ("system", """You are an AI Education Assistant. Your task is to generate a concise, accurate, and age-appropriate explanation about the user's topic.

                      - **Length:** The explanation must be between 80 to 150 words.
                      - **Language:** Write the explanation entirely in English.
                      - **Tone:** Use a {tone} tone. The friendly tone is for younger students. The formal tone is for older students.
                      - **Context:** Use the provided context (RAG) AND your internal knowledge base. Prioritize accuracy.
                      - **Output:** Respond ONLY with a single JSON object that strictly adheres to the following schema. DO NOT include any other text or markdown outside the JSON object: {response_json}

                        RAG Context: {full_retrieved_context}
            """),
          ("human", "Generate an explanation for the topic: '{topic}'")
      ]
  )

GoogleTranslator




In [ ]:
# Config
lang = {
    "English": "en",
    "Tamil": "ta",
    "Hindi": "hi"
}

In [ ]:
def generate_explanation(topic: str, language: str = "English", tone: str = "formal"):
    """
    Generates a written explanation and its spoken audio for a given topic using ParlerTTS.
    """
    if language not in LANGUAGE_MAP: # Check against LANGUAGE_MAP for supported languages
        raise ValueError(f"Unsupported language for LLM prompt: {language}. Choose from: {list(LANGUAGE_MAP.keys())}")


    # 1. RAG Retrieval
    try:
        # Retrieve context from PDF (ChromaDB)
        pdf_docs = pdf_retriever.invoke(topic)
        pdf_context = "\n".join([d.page_content for d in pdf_docs])

        # Retrieve context from Wikipedia
        wiki_docs = wiki_retriever.invoke(topic)
        wiki_context = "\n".join([d.page_content for d in wiki_docs])

        full_context = f"PDF Context:\n---\n{pdf_context}\n\nWikipedia Context:\n---\n{wiki_context}"

    except Exception as e:
        print(f"Warning: RAG retrieval failed: {e}. Relying on LLM knowledge only.")
        full_context = "No specific RAG context could be retrieved."

    # 2. LLM Generation Chain
    explanation_chain = (
        {
            "full_retrieved_context": itemgetter("full_context"),
            "topic": itemgetter("topic"),
            "tone": itemgetter("tone"),
            "response_json": lambda x: json.dumps({"text_explanation": RESPONSE_JSON["text_explanation"]}) # Pass simple JSON schema

        }
        | prompt
        | llm
        | JsonOutputParser()
    )

    try:
        llm_input = {
            "full_context": full_context,
            "topic": topic,
            "tone": tone

        }

        llm_output = explanation_chain.invoke(llm_input)
        text_explanation = llm_output.get("text_explanation", "Error: LLM failed to generate text_explanation.")
        print(f"LLM Explanation Generated:\n{text_explanation}\n")

    except Exception as e:
        print(f"Error during LLM generation: {e}")
        return {"error": "LLM generation failed.", "audio_url": "Error: LLM generation failed."}

    # 3.Google Translator
    try:
        google_trans = GoogleTranslator(source='auto', target=lang[language]).translate( text_explanation)
        print(f"Google Translate (ta): {google_trans}")
    except Exception as e:
        print(f"Google Translate error: {e}")

    # 4.ParlerTTs

    # Generate audio for long text
    print("Starting long text TTS generation...")
    final_audio = generate_long_audio(model, tokenizer, description_tokenizer, google_trans, DESCRIPTION[language], max_words=40)

    audio_filename = "long_indic_tts_out.wav"
    # Save the final audio
    sf.write(audio_filename, final_audio, model.config.sampling_rate)
    print("Audio generation completed! Saved as 'long_indic_tts_out.wav'")

    # 5. Final Output
    return {
        "text_explanation": google_trans,
        "audio_url": audio_filename # The path/url will be the local Colab file path
    }

In [ ]:
# --- EXAMPLE 1: English (Friendly Tone, using your target output) ---
# Assuming your PDF contains some English content relevant to science/gravity.
topic_1 = "The Journey of Christopher Columbus"
result_1 = generate_explanation(topic=topic_1, language="English", tone="friendly")

print("\n\n--- Output 1 (English) ---")
print(json.dumps(result_1, indent=2))

# Display the audio link for easy listening in Colab
from IPython.display import Audio
if "output" in result_1.get("audio_url", ""):
    display(Audio(result_1["audio_url"]))

In [ ]:
# --- EXAMPLE 2: Hindi (Formal Tone, for a different topic) ---
topic_2 = "Photosynthesis"
result_2 = generate_explanation(topic=topic_2, language="Hindi", tone="formal")

print("\n\n--- Output 2 (Hindi) ---")
print(json.dumps(result_2, indent=2, ensure_ascii=False)) # Use ensure_ascii=False to display Hindi text properly

# Display the audio link for easy listening in Colab
if "output" in result_2.get("audio_url", ""):
    display(Audio(result_2["audio_url"]))

In [ ]:
# --- EXAMPLE 3: Tamil (Formal Tone, referencing the PDF/RAG data) ---
# Replace 'Exploring the Seas' with a topic you know is in your Class 10 English PDF
topic_3 = "thirukkural"
result_3 = generate_explanation(topic=topic_3, language="Tamil", tone="formal")

print("\n\n--- Output 3 (Tamil) ---")
print(json.dumps(result_3, indent=2, ensure_ascii=False))

# Display the audio link for easy listening in Colab
if "output" in result_3.get("audio_url", ""):
    display(Audio(result_3["audio_url"]))

In [ ]:
!pip show langchain_groq pypdf langchain_community langchain chromadb groq coqui-tts wikipedia